In [8]:
import time
import numpy as np

class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.pmt_positions = self.generate_pmt_positions()
        self.pmt_angular_radius = np.arcsin(self.r / self.R)

    def generate_pmt_positions(self):
        positions = []
        phi = np.pi * (np.sqrt(5.0) - 1.0)
        for i in range(self.N):
            y = 1.0 - (i / float(self.N - 1)) * 2.0
            radius_at_y = np.sqrt(1.0 - y * y)
            theta = phi * i
            x = np.cos(theta) * radius_at_y
            z = np.sin(theta) * radius_at_y
            pos = np.array([x, y, z]) * self.R
            positions.append(pos)
        return np.array(positions)

    # векторы
    def check_hits_vectorized(self, hit_points):
        norm_hits = hit_points / np.linalg.norm(hit_points, axis=1, keepdims=True)
        norm_pmts = self.pmt_positions / self.R
        cos_angles = np.dot(norm_hits, norm_pmts.T)
        cos_angles = np.clip(cos_angles, -1.0, 1.0)
        angles = np.arccos(cos_angles)
        
        hits_mask = angles <= self.pmt_angular_radius
        hit_indices = np.argmax(hits_mask, axis=1)
        has_hit = np.any(hits_mask, axis=1)
        hit_indices[~has_hit] = -1
        
        valid_hits = hit_indices[hit_indices != -1]
        return np.bincount(valid_hits, minlength=self.N)

    # скаляры
    def check_hit_single(self, hit_point):
        norm_hit = hit_point / np.linalg.norm(hit_point)
        for idx, pmt_pos in enumerate(self.pmt_positions):
            norm_pmt = pmt_pos / self.R
            cos_angle = np.clip(np.dot(norm_hit, norm_pmt), -1.0, 1.0)
            if np.arccos(cos_angle) <= self.pmt_angular_radius:
                return idx
        return -1


class SmartSource:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons(self, energy, pos=(0.0, 0.0, 0.0)):
        n_photons = int(energy * self.Y)
        phi = np.random.uniform(0, 2 * np.pi, n_photons)
        costheta = np.random.uniform(-1.0, 1.0, n_photons)
        sintheta = np.sqrt(1.0 - costheta**2)
        
        directions = np.column_stack((sintheta * np.cos(phi), sintheta * np.sin(phi), costheta))
        positions = np.tile(pos, (n_photons, 1))
        return positions, directions

detector = Detector()
source = SmartSource()
energy = 1.0  
r0, directions = source.generate_photons(energy)

b = 2 * np.sum(r0 * directions, axis=1)
c = np.sum(r0**2, axis=1) - detector.R**2
t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
hit_points = r0 + directions * t[:, np.newaxis]

# время вектора
start_time = time.time()
pmt_hits_vec = detector.check_hits_vectorized(hit_points)
time_vectorized = time.time() - start_time
print( время := f"1. NumPy векторизация: {time_vectorized:.6f} сек.")

# время скаляра
start_time = time.time()
pmt_hits_loop = np.zeros(detector.N, dtype=int)
for hp in hit_points:
    idx = detector.check_hit_single(hp)
    if idx != -1:
        pmt_hits_loop[idx] += 1
time_loop = time.time() - start_time
print(f"2. Поштучный цикл: {time_loop:.4f} сек.")

# оценка выигрыша
speedup = time_loop / time_vectorized
print(f"\nВыигрыш во времени от использования numpy: примерно в {speedup:.1f} раз быстрее")

1. NumPy векторизация: 0.009617 сек.
2. Поштучный цикл: 6.5612 сек.

Выигрыш во времени от использования numpy: примерно в 682.3 раз быстрее
